# Step 4 - Model Development

**Baseline: SARIMAX with exogenous regressors.**

Cleaned dataset only - no engineered features.

Metrics are MAPE and RMSE per the brief, with WAPE alongside because MAPE is
undefined on zero-consumption days (12.2% of rows). Selection is on the validation
split; the test period is not touched.

In [ ]:
import sys
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

# Add src folder to Python path
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project Root:", PROJECT_ROOT)

In [ ]:
import warnings

import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

from mig_cement.config import settings

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

TRAIN_END, VAL_END = "2024-06-30", "2024-09-30"
TARGET = "y"
HORIZON_WEEKS = 8   # the forecast horizon the brief specifies

## 1. Load the cleaned dataset

In [ ]:
clean = pd.read_parquet(settings.interim_dir / "operations_clean.parquet")
clean["date"] = pd.to_datetime(clean["date"])
clean = clean.sort_values(["site_id", "date"]).reset_index(drop=True)

print("shape:", clean.shape)
print("sites:", clean.site_id.nunique(), "| dates:", clean.date.nunique())
print("range:", clean.date.min().date(), "->", clean.date.max().date())

## 2. Handle NaNs

Rows with NaN are removed, scoped to the columns this model uses.

A blanket `dropna()` would be a trap: `cover_days` is NaN exactly where
`consumed_tonnes == 0`, so it silently deletes every zero-consumption day - the
rain-blocked pours and stockouts, which are the hard cases.

In [ ]:
print("NaN counts:")
print(clean.isna().sum()[lambda s: s > 0].to_string())
print("\nblanket dropna would give:", clean.dropna().shape,
      f"({100*(1-len(clean.dropna())/len(clean)):.1f}% lost)")
print("  zero-y rows before:", int((clean[TARGET] == 0).sum()),
      "| after:", int((clean.dropna()[TARGET] == 0).sum()))

In [ ]:
EXOG = ["planned_pour_tonnes", "rain_mm", "avg_temp_c", "opening_inventory_tonnes"]

before = len(clean)
clean = clean.dropna(subset=[TARGET] + EXOG).reset_index(drop=True)
print(f"rows: {before:,} -> {len(clean):,}")
print(f"zero-y rows retained: {int((clean[TARGET] == 0).sum()):,} "
      f"({(clean[TARGET] == 0).mean():.1%})")

### Why these four regressors

Of the 22 columns in the cleaned panel, most cannot be used:

- **target-derived / leaky**: `consumed_tonnes`, `served_tonnes`,
  `closing_inventory_tonnes`, `cover_days`, `silo_utilisation`, `was_constrained`,
  `unmet_tonnes`, `induced_shortfall`
- **not knowable at forecast time**: `deliveries_tonnes`, `received_tonnes`,
  `rejected_delivery_tonnes`
- **constant within each site**: `silo_capacity`, `region`, `behavior` - models are
  fitted per site, so these have no within-series variance and are collinear with
  the intercept
- **keys**: `date`, `site_id`, `cement_type`

## 3. Train / validation / test split

Chronological. The test period is held back.

In [ ]:
d = clean["date"]
train = clean[d <= TRAIN_END]
val = clean[(d > TRAIN_END) & (d <= VAL_END)]
test = clean[d > VAL_END]

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:6s} {len(part):6,} rows  {part.date.min().date()} -> {part.date.max().date()}")

## 4. Model configuration

`d = 0` because the series are stationary. `seasonal_order = (0,0,0,0)` because
Step 3 tested weekly, monthly and annual seasonality per region against a shuffled
null and found none - seasonal terms would fit noise.

In [ ]:
adf = pd.Series({s: adfuller(g[TARGET])[1] for s, g in clean.groupby("site_id")})
print(f"ADF p-values across {len(adf)} sites: max = {adf.max():.2e}")
print(f"sites rejecting a unit root at 1%: {(adf < 0.01).sum()} / {len(adf)}")
print("\n-> d = 0")

In [ ]:
# order chosen by mean AIC across a sample of sites
GRID = [(1, 0, 0), (0, 0, 1), (1, 0, 1), (2, 0, 1), (2, 0, 2)]
aic = {}
for o in GRID:
    scores = []
    for site in sorted(train.site_id.unique())[:5]:
        g = train[train.site_id == site].set_index("date")
        try:
            scores.append(SARIMAX(g[TARGET], exog=g[EXOG], order=o,
                                  seasonal_order=(0, 0, 0, 0)).fit(disp=False).aic)
        except Exception:
            pass
    aic[str(o)] = np.mean(scores)

aic = pd.Series(aic).sort_values()
print(aic.round(1).to_string())
ORDER = (2, 0, 2)
print("\nselected:", ORDER)

## 5. Train the model

Fitted on one site first, in the plainest form.

In [ ]:
SITE = "SITE_001"

y_train = train[train.site_id == SITE].set_index("date")[TARGET]
x_train = train[train.site_id == SITE].set_index("date")[EXOG]
y_val = val[val.site_id == SITE].set_index("date")[TARGET]
x_val = val[val.site_id == SITE].set_index("date")[EXOG]

print(f"{SITE}: train {y_train.shape[0]} rows, val {y_val.shape[0]} rows, "
      f"{x_train.shape[1]} exogenous regressors")

In [ ]:
model = SARIMAX(
    y_train,
    exog=x_train,
    order=ORDER,
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=True,
)

results = model.fit(disp=False)
results.summary()

`enforce_stationarity=True` is deliberate. With it set to `False`, an unstable
AR root produced forecasts that diverged across the 92-day validation window -
one site reached an RMSE of 1.96e20. The series are stationary, so the constraint
costs nothing.

## 6. Predict

`y_val` / `X_val` below are the **validation** window (Jul-Sep 2024).
Oct-Dec 2024 stays held back and is not touched in this notebook.

In [ ]:
y_pred = results.predict(start=y_val.index[0], end=y_val.index[-1], exog=x_val)
y_pred = y_pred.clip(lower=0)
y_pred

## 7. Metrics

`sklearn.metrics.mean_absolute_percentage_error` returns a **fraction, not a
percentage** - a returned value of 15 means 1500%, not 15%.

It is also undefined when the actual is zero. 12.2% of site-days have no pour, and
on those rows sklearn divides by a tiny epsilon, so a handful of rows can dominate
the whole average. MAPE is therefore computed on non-zero actuals only, with the
raw sklearn figure shown alongside so the gap is visible.

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

nz = y_val != 0

mape_raw = mean_absolute_percentage_error(y_val, y_pred)
mape_nonzero = mean_absolute_percentage_error(y_val[nz], y_pred[nz])
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(f"{SITE}")
print(f"  zero-actual days      {int((~nz).sum())} of {len(y_val)}")
print(f"  MAPE (sklearn raw)    {mape_raw:.4f}  = {mape_raw*100:,.0f}%")
print(f"  MAPE (non-zero only)  {mape_nonzero:.4f}  = {mape_nonzero*100:.1f}%")
print(f"  RMSE                  {rmse:.3f} t")

## 8. Fit all 30 sites and predict

In [ ]:
models, preds = {}, []

for site, g_tr in train.groupby("site_id"):
    g_tr = g_tr.set_index("date")
    g_te = val[val.site_id == site].sort_values("date").set_index("date")

    y_tr, X_tr = g_tr[TARGET], g_tr[EXOG]
    y_te, X_te = g_te[TARGET], g_te[EXOG]

    try:
        res = SARIMAX(
            y_tr,
            exog=X_tr,
            order=ORDER,
            seasonal_order=(0, 0, 0, 0),
            enforce_stationarity=True,
        ).fit(disp=False)
        pred = res.predict(start=y_te.index[0], end=y_te.index[-1], exog=X_te).clip(lower=0)
    except Exception:
        res, pred = None, pd.Series(np.nan, index=y_te.index)

    models[site] = res
    preds.append(pd.DataFrame({"date": y_te.index, "site_id": site,
                               "actual": y_te.values, "pred": pred.values}))

fc = pd.concat(preds, ignore_index=True).dropna(subset=["pred"])
converged = sum(m is not None for m in models.values())
print(f"sites: {len(models)} | converged: {converged} | failed: {len(models) - converged}")
print(f"{len(fc):,} predictions | {fc.date.nunique()} dates x {fc.site_id.nunique()} sites")

## 9. Daily error, per site

One row per site-day: absolute error, squared error, and percentage error where the
actual is non-zero.

In [ ]:
fc["abs_err"] = (fc.actual - fc.pred).abs()
fc["sq_err"] = (fc.actual - fc.pred) ** 2
fc["pct_err"] = np.where(fc.actual != 0, fc.abs_err / fc.actual, np.nan)

print(f"{len(fc):,} site-days | {fc.pct_err.isna().sum():,} with zero actual (no MAPE)")
fc.head(10).round(3)

In [ ]:
per_site = pd.DataFrame({
    "n_days": fc.groupby("site_id").size(),
    "zero_days": fc.groupby("site_id").pct_err.apply(lambda s: int(s.isna().sum())),
    "mean_actual": fc.groupby("site_id").actual.mean(),
    "MAPE": fc.groupby("site_id").pct_err.mean(),
    "RMSE": fc.groupby("site_id").sq_err.mean() ** 0.5,
}).sort_values("MAPE", ascending=False)

print(f"MAPE across sites: best {per_site.MAPE.min():.1%} | "
      f"median {per_site.MAPE.median():.1%} | worst {per_site.MAPE.max():.1%}")
per_site.round(4)

## 10. Daily error, per calendar date across all sites

In [ ]:
per_day = pd.DataFrame({
    "n_sites": fc.groupby("date").size(),
    "zero_sites": fc.groupby("date").pct_err.apply(lambda s: int(s.isna().sum())),
    "mean_actual": fc.groupby("date").actual.mean(),
    "mean_pred": fc.groupby("date").pred.mean(),
    "MAPE": fc.groupby("date").pct_err.mean(),
    "RMSE": fc.groupby("date").sq_err.mean() ** 0.5,
})

print(f"{len(per_day)} days | MAPE best {per_day.MAPE.min():.1%} | "
      f"median {per_day.MAPE.median():.1%} | worst {per_day.MAPE.max():.1%}")
per_day.round(4)

In [ ]:
per_day_h = per_day.reset_index()
per_day_h["horizon_day"] = np.arange(1, len(per_day_h) + 1)
per_day_h["week"] = ((per_day_h.horizon_day - 1) // 7) + 1

by_week = per_day_h.groupby("week").agg(
    days=("horizon_day", "size"), MAPE=("MAPE", "mean"), RMSE=("RMSE", "mean")).head(8)
print("error by forecast week (the 8-week horizon in the brief):")
by_week.round(4)

## 11. Overall, against baselines

In [ ]:
def score(y_true, y_pred_):
    y_true, y_pred_ = np.asarray(y_true, float), np.asarray(y_pred_, float)
    nzm = y_true != 0
    return {
        "MAPE": mean_absolute_percentage_error(y_true[nzm], y_pred_[nzm]),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred_)),
        "WAPE": np.abs(y_true - y_pred_).sum() / np.abs(y_true).sum(),
        "MAE": np.abs(y_true - y_pred_).mean(),
        "bias": (y_pred_ - y_true).mean(),
    }


rows = [
    {**score(val[TARGET], val["planned_pour_tonnes"]), "model": "baseline: planned_pour"},
    {**score(val[TARGET], np.full(len(val), train[TARGET].mean())), "model": "baseline: train mean"},
    {**score(fc.actual, fc.pred), "model": "SARIMAX (cleaned data)"},
]
results_tbl = pd.DataFrame(rows).set_index("model")[["MAPE", "RMSE", "WAPE", "MAE", "bias"]]
results_tbl.round(4)

---

# Experiment 2 - Weekly Aggregation

Same cleaned dataset, same regressors, same model. The only change is the grain:
each site-day is aggregated to a site-week.

Consumption and planned pour are **summed**, weather is **averaged**, and opening
inventory takes the **first** value of the week.

In [ ]:
AGG = {
    "y": "sum",
    "planned_pour_tonnes": "sum",
    "rain_mm": "mean",
    "avg_temp_c": "mean",
    "opening_inventory_tonnes": "first",
}

grp = clean.set_index("date").groupby("site_id").resample("W")
weekly = grp.agg(AGG).reset_index()
weekly["n_days"] = grp.size().values

# Drop partial weeks. resample("W") opens and closes the series with buckets that
# hold fewer than 7 days - the final one covers only 30-31 Dec, and averages 49.5 t
# against 162.6 t for a full week. Left in, it reads as a demand collapse.
partial = weekly.n_days < 7
print(f"partial weeks dropped: {partial.sum()} of {len(weekly)}")
print(weekly.loc[partial, ["date", "n_days", "y"]].groupby("date")
             .agg(sites=("n_days", "size"), days=("n_days", "first"),
                  mean_y=("y", "mean")).round(1).to_string())

weekly = (weekly[~partial]
          .drop(columns="n_days")
          .dropna(subset=["y"])
          .sort_values(["site_id", "date"])
          .reset_index(drop=True))

print("\ndaily :", clean.shape, "| mean y", round(clean.y.mean(), 2), "t",
      "| zero rows", f"{(clean.y == 0).mean():.1%}")
print("weekly:", weekly.shape, "| mean y", round(weekly.y.mean(), 2), "t",
      "| zero rows", f"{(weekly.y == 0).mean():.1%}")
weekly.head()

Aggregation removes the zero-consumption problem entirely: no site goes a full
week without pouring, so MAPE becomes well-defined on every row.

## Split

In [ ]:
dw = weekly["date"]
train_w = weekly[dw <= TRAIN_END]
val_w = weekly[(dw > TRAIN_END) & (dw <= VAL_END)]
test_w = weekly[dw > VAL_END]

# The brief asks for forecasts up to 8 weeks ahead, so scoring is capped at that
# horizon. Beyond it the model is being judged on something it does not promise.
val_w = val_w.groupby("site_id").head(HORIZON_WEEKS).reset_index(drop=True)
test_w = test_w.groupby("site_id").head(HORIZON_WEEKS).reset_index(drop=True)

for name, part in [("train", train_w), ("val", val_w), ("test", test_w)]:
    print(f"{name:6s} {len(part):5,} rows  {part.date.min().date()} -> {part.date.max().date()}"
          f"  ({part.groupby('site_id').size().mean():.0f} weeks per site)")

## Order selection

In [ ]:
aic_w = {}
for o in GRID:
    scores = []
    for site in sorted(train_w.site_id.unique())[:5]:
        g = train_w[train_w.site_id == site].set_index("date").asfreq("W")
        try:
            scores.append(SARIMAX(g[TARGET], exog=g[EXOG], order=o,
                                  seasonal_order=(0, 0, 0, 0)).fit(disp=False).aic)
        except Exception:
            pass
    aic_w[str(o)] = np.mean(scores) if scores else np.nan

aic_w = pd.Series(aic_w).sort_values()
print(aic_w.round(1).to_string())
ORDER_W = eval(aic_w.index[0])
print("\nselected:", ORDER_W)

## Train the model

In [ ]:
y_train_w = train_w[train_w.site_id == SITE].set_index("date").asfreq("W")[TARGET]
x_train_w = train_w[train_w.site_id == SITE].set_index("date").asfreq("W")[EXOG]
y_val_w = val_w[val_w.site_id == SITE].set_index("date").asfreq("W")[TARGET]
x_val_w = val_w[val_w.site_id == SITE].set_index("date").asfreq("W")[EXOG]

print(f"{SITE}: train {len(y_train_w)} weeks, val {len(y_val_w)} weeks")

In [ ]:
model_w = SARIMAX(
    y_train_w,
    exog=x_train_w,
    order=ORDER_W,
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=True,
)

results_w = model_w.fit(disp=False)
results_w.summary()

## Predict

In [ ]:
y_pred_w = results_w.predict(start=y_val_w.index[0], end=y_val_w.index[-1], exog=x_val_w)
y_pred_w = y_pred_w.clip(lower=0)

print(f"{SITE}")
print(f"  MAPE {mean_absolute_percentage_error(y_val_w, y_pred_w):.4f}"
      f"  = {mean_absolute_percentage_error(y_val_w, y_pred_w)*100:.1f}%")
print(f"  RMSE {np.sqrt(mean_squared_error(y_val_w, y_pred_w)):.3f} t")
y_pred_w

## Fit all 30 sites

In [ ]:
models_w, preds_w = {}, []

for site, g_tr in train_w.groupby("site_id"):
    g_tr = g_tr.set_index("date").asfreq("W")
    g_te = val_w[val_w.site_id == site].sort_values("date").set_index("date").asfreq("W")

    try:
        res = SARIMAX(
            g_tr[TARGET],
            exog=g_tr[EXOG],
            order=ORDER_W,
            seasonal_order=(0, 0, 0, 0),
            enforce_stationarity=True,
        ).fit(disp=False)
        pred = res.predict(start=g_te.index[0], end=g_te.index[-1],
                           exog=g_te[EXOG]).clip(lower=0)
    except Exception:
        res, pred = None, pd.Series(np.nan, index=g_te.index)

    models_w[site] = res
    preds_w.append(pd.DataFrame({"date": g_te.index, "site_id": site,
                                 "actual": g_te[TARGET].values, "pred": pred.values}))

fc_w = pd.concat(preds_w, ignore_index=True).dropna(subset=["pred"])
converged_w = sum(m is not None for m in models_w.values())
print(f"sites: {len(models_w)} | converged: {converged_w} | failed: {len(models_w) - converged_w}")
print(f"{len(fc_w):,} predictions | {fc_w.date.nunique()} weeks x {fc_w.site_id.nunique()} sites")

## Weekly error, per calendar week across all sites

In [ ]:
fc_w["abs_err"] = (fc_w.actual - fc_w.pred).abs()
fc_w["sq_err"] = (fc_w.actual - fc_w.pred) ** 2
fc_w["pct_err"] = np.where(fc_w.actual != 0, fc_w.abs_err / fc_w.actual, np.nan)

per_week = pd.DataFrame({
    "n_sites": fc_w.groupby("date").size(),
    "zero_sites": fc_w.groupby("date").pct_err.apply(lambda s: int(s.isna().sum())),
    "mean_actual": fc_w.groupby("date").actual.mean(),
    "mean_pred": fc_w.groupby("date").pred.mean(),
    "MAPE": fc_w.groupby("date").pct_err.mean(),
    "RMSE": fc_w.groupby("date").sq_err.mean() ** 0.5,
})

print(f"{len(per_week)} weeks | MAPE best {per_week.MAPE.min():.1%} | "
      f"median {per_week.MAPE.median():.1%} | worst {per_week.MAPE.max():.1%}")
per_week.round(4)

## Weekly error, per site

In [ ]:
per_site_w = pd.DataFrame({
    "n_weeks": fc_w.groupby("site_id").size(),
    "mean_actual": fc_w.groupby("site_id").actual.mean(),
    "MAPE": fc_w.groupby("site_id").pct_err.mean(),
    "RMSE": fc_w.groupby("site_id").sq_err.mean() ** 0.5,
}).sort_values("MAPE", ascending=False)

print(f"MAPE across sites: best {per_site_w.MAPE.min():.1%} | "
      f"median {per_site_w.MAPE.median():.1%} | worst {per_site_w.MAPE.max():.1%}")
per_site_w.round(4)

## Daily vs weekly

In [ ]:
rows_w = [
    {**score(val_w[TARGET], val_w["planned_pour_tonnes"]), "model": "planned_pour (weekly)"},
    {**score(fc_w.actual, fc_w.pred), "model": "SARIMAX (weekly)"},
]
comparison = pd.concat([results_tbl, pd.DataFrame(rows_w).set_index("model")[
    ["MAPE", "RMSE", "WAPE", "MAE", "bias"]]])
comparison.round(4)

**RMSE is not comparable across grains.** A weekly total is roughly seven times a
daily value, so its RMSE is larger by construction - that is arithmetic, not a
worse model. MAPE and WAPE are scale-relative and can be compared.

Aggregation also makes the problem mechanically easier: day-to-day noise cancels
when summed, and the 12.2% of zero-pour days disappear. A lower weekly MAPE is
therefore partly a real gain in usable accuracy and partly an easier question. The
figure that carries meaning is the **gap between SARIMAX and `planned_pour` at each
grain**, since both face the same conditions.

Weekly is also the grain MIG actually reorders on, which is the practical argument
for it regardless of the arithmetic.

---

# Experiment 3 - Engineered Features, Weekly Aggregation

Weekly grain again, but with the engineered features from notebook 03 as regressors
instead of the four raw columns.

Aggregation is per feature type rather than one blanket rule - summing a flag and
summing a tonnage mean different things:

| feature | rule | meaning at weekly grain |
|---|---|---|
| `planned_pour_tonnes` | sum | tonnes scheduled that week |
| `planned_pour_next_7/14` | last | schedule looking forward from week end |
| `pour_blocked_rain` | **sum** | days lost to rain that week |
| `frost` | **sum** | frost days that week |
| `rain_mm`, `avg_temp_c` | mean | average conditions |
| `opening_inventory_tonnes`, `inventory_vs_capacity`, `headroom_tonnes` | first | position entering the week |
| `cover_days_7`, `days_since_planned_pour` | first | state entering the week |

`pour_blocked_rain` is the interesting one: as a daily 0/1 flag it marks a single
lost day, but summed it becomes "how many pour days this week were rained off",
which is a genuinely different and more useful quantity.

Target lags and rolling means are **excluded**. SARIMAX already models the
autoregressive structure through its AR terms, so passing lagged target values as
exogenous regressors double-counts them and destabilises the fit.

In [ ]:
feats = pd.read_parquet(settings.processed_dir / "operations_feature_engineered.parquet")
feats["date"] = pd.to_datetime(feats["date"])
feats = feats.sort_values(["site_id", "date"]).reset_index(drop=True)
print("engineered daily matrix:", feats.shape)

AGG_ENG = {
    "y": "sum",
    "planned_pour_tonnes": "sum",
    "planned_pour_next_7": "last",
    "planned_pour_next_14": "last",
    "pour_blocked_rain": "sum",
    "frost": "sum",
    "rain_mm": "mean",
    "avg_temp_c": "mean",
    "opening_inventory_tonnes": "first",
    "inventory_vs_capacity": "first",
    "headroom_tonnes": "first",
    "cover_days_7": "first",
    "days_since_planned_pour": "first",
}

grp_e = feats.set_index("date").groupby("site_id").resample("W")
weekly_eng = grp_e.agg(AGG_ENG).reset_index()
weekly_eng["n_days"] = grp_e.size().values

EXOG_ENG = [c for c in AGG_ENG if c != "y"]

before = len(weekly_eng)
partial_e = weekly_eng.n_days < 7
weekly_eng = (weekly_eng[~partial_e]
              .drop(columns="n_days")
              .dropna(subset=["y"] + EXOG_ENG)
              .sort_values(["site_id", "date"])
              .reset_index(drop=True))
print(f"partial weeks dropped: {partial_e.sum()}")
print(f"weekly engineered: {before:,} -> {len(weekly_eng):,} rows")
print(f"{len(EXOG_ENG)} regressors (weekly used 4)")
weekly_eng.head()

In [ ]:
print("what the aggregated flags look like:")
print(weekly_eng[["pour_blocked_rain", "frost"]].describe().loc[
    ["mean", "50%", "max"]].round(2).to_string())
print(f"\nweeks with at least one rained-off day: "
      f"{(weekly_eng.pour_blocked_rain > 0).mean():.1%}")
print(f"weeks with at least one frost day:       {(weekly_eng.frost > 0).mean():.1%}")

## Split

In [ ]:
de = weekly_eng["date"]
train_e = weekly_eng[de <= TRAIN_END]
val_e = weekly_eng[(de > TRAIN_END) & (de <= VAL_END)]
test_e = weekly_eng[de > VAL_END]

val_e = val_e.groupby("site_id").head(HORIZON_WEEKS).reset_index(drop=True)
test_e = test_e.groupby("site_id").head(HORIZON_WEEKS).reset_index(drop=True)

for name, part in [("train", train_e), ("val", val_e), ("test", test_e)]:
    print(f"{name:5s} {len(part):5,} rows  {part.date.min().date()} -> {part.date.max().date()}"
          f"  ({part.groupby('site_id').size().mean():.0f} weeks per site)")

## Order selection

In [ ]:
aic_e = {}
for o in GRID:
    scores = []
    for site in sorted(train_e.site_id.unique())[:5]:
        g = train_e[train_e.site_id == site].set_index("date").asfreq("W")
        try:
            scores.append(SARIMAX(g[TARGET], exog=g[EXOG_ENG], order=o,
                                  seasonal_order=(0, 0, 0, 0)).fit(disp=False).aic)
        except Exception:
            pass
    aic_e[str(o)] = np.mean(scores) if scores else np.nan

aic_e = pd.Series(aic_e).sort_values()
print(aic_e.round(1).to_string())
ORDER_E = eval(aic_e.index[0])
print("\nselected:", ORDER_E)

## Train the model

In [ ]:
y_train_e = train_e[train_e.site_id == SITE].set_index("date").asfreq("W")[TARGET]
x_train_e = train_e[train_e.site_id == SITE].set_index("date").asfreq("W")[EXOG_ENG]
y_val_e = val_e[val_e.site_id == SITE].set_index("date").asfreq("W")[TARGET]
x_val_e = val_e[val_e.site_id == SITE].set_index("date").asfreq("W")[EXOG_ENG]

print(f"{SITE}: train {len(y_train_e)} weeks, val {len(y_val_e)} weeks, "
      f"{x_train_e.shape[1]} regressors")

In [ ]:
model_e = SARIMAX(
    y_train_e,
    exog=x_train_e,
    order=ORDER_E,
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=True,
)

results_e = model_e.fit(disp=False)
results_e.summary()

## Predict

In [ ]:
y_pred_e = results_e.predict(start=y_val_e.index[0], end=y_val_e.index[-1], exog=x_val_e)
y_pred_e = y_pred_e.clip(lower=0)

print(f"{SITE}")
print(f"  MAPE {mean_absolute_percentage_error(y_val_e, y_pred_e)*100:.1f}%")
print(f"  RMSE {np.sqrt(mean_squared_error(y_val_e, y_pred_e)):.3f} t")
y_pred_e

## Fit all 30 sites

In [ ]:
models_e, preds_e = {}, []

for site, g_tr in train_e.groupby("site_id"):
    g_tr = g_tr.set_index("date").asfreq("W")
    g_va = val_e[val_e.site_id == site].sort_values("date").set_index("date").asfreq("W")

    try:
        res = SARIMAX(
            g_tr[TARGET],
            exog=g_tr[EXOG_ENG],
            order=ORDER_E,
            seasonal_order=(0, 0, 0, 0),
            enforce_stationarity=True,
        ).fit(disp=False)
        pred = res.predict(start=g_va.index[0], end=g_va.index[-1],
                           exog=g_va[EXOG_ENG]).clip(lower=0)
    except Exception:
        res, pred = None, pd.Series(np.nan, index=g_va.index)

    models_e[site] = res
    preds_e.append(pd.DataFrame({"date": g_va.index, "site_id": site,
                                 "actual": g_va[TARGET].values, "pred": pred.values}))

fc_e = pd.concat(preds_e, ignore_index=True).dropna(subset=["pred"])
conv_e = sum(m is not None for m in models_e.values())
print(f"sites: {len(models_e)} | converged: {conv_e} | failed: {len(models_e) - conv_e}")
print(f"{len(fc_e):,} predictions | {fc_e.date.nunique()} weeks x {fc_e.site_id.nunique()} sites")

## Weekly error, per calendar week across all sites

In [ ]:
fc_e["abs_err"] = (fc_e.actual - fc_e.pred).abs()
fc_e["sq_err"] = (fc_e.actual - fc_e.pred) ** 2
fc_e["pct_err"] = np.where(fc_e.actual != 0, fc_e.abs_err / fc_e.actual, np.nan)

per_week_e = pd.DataFrame({
    "n_sites": fc_e.groupby("date").size(),
    "mean_actual": fc_e.groupby("date").actual.mean(),
    "mean_pred": fc_e.groupby("date").pred.mean(),
    "MAPE": fc_e.groupby("date").pct_err.mean(),
    "RMSE": fc_e.groupby("date").sq_err.mean() ** 0.5,
})

print(f"{len(per_week_e)} weeks | MAPE best {per_week_e.MAPE.min():.1%} | "
      f"median {per_week_e.MAPE.median():.1%} | worst {per_week_e.MAPE.max():.1%}")
per_week_e.round(4)

## Weekly error, per site

In [ ]:
per_site_e = pd.DataFrame({
    "n_weeks": fc_e.groupby("site_id").size(),
    "mean_actual": fc_e.groupby("site_id").actual.mean(),
    "MAPE": fc_e.groupby("site_id").pct_err.mean(),
    "RMSE": fc_e.groupby("site_id").sq_err.mean() ** 0.5,
}).sort_values("MAPE", ascending=False)

print(f"MAPE across sites: best {per_site_e.MAPE.min():.1%} | "
      f"median {per_site_e.MAPE.median():.1%} | worst {per_site_e.MAPE.max():.1%}")
per_site_e.round(4)

## All experiments compared

In [ ]:
final = pd.concat([
    comparison,
    pd.DataFrame([{**score(fc_e.actual, fc_e.pred),
                   "model": "SARIMAX (engineered, weekly)"}]).set_index("model")[
        ["MAPE", "RMSE", "WAPE", "MAE", "bias"]],
])
final.round(4)

In [ ]:
weekly_only = final.loc[["planned_pour (weekly)", "SARIMAX (weekly)",
                         "SARIMAX (engineered, weekly)"]]
weekly_only.assign(**{
    "MAPE vs raw-feature model": (
        weekly_only.MAPE / final.loc["SARIMAX (weekly)", "MAPE"] - 1
    ).map(lambda x: f"{x:+.1%}"),
    "meets MAPE <= 15%": weekly_only.MAPE.le(0.15).map({True: "yes", False: "no"}),
}).round(4)

## Notes

- Cleaned dataset only, no engineered features.
- `deliveries_tonnes`, `closing_inventory_tonnes` and `silo_capacity` are excluded
  from the regressors. The first two satisfy
  `consumed = opening + deliveries - closing` exactly, so including them lets the
  model reproduce the target instead of forecasting it; the third is constant
  within a site and makes the covariance matrix singular.
- MAPE is computed on non-zero actuals; the raw sklearn value is in section 7.
- Weather regressors use actual validation values, which flatters the result.
- The Oct-Dec 2024 test split is untouched.

# Global Machine Learning Models
## Random Forest Regressor

In [ ]:
print(clean.columns.tolist())


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

## Prepare Features and Target

The machine learning model is trained using the pooled dataset across all sites. Predictor variables include operational, weather, inventory, and site-level characteristics that are available before or at the time a forecast is made.

The target variable is **y**, representing cement demand.

In [ ]:
# Review fix: cover_days, silo_utilisation and deliveries_tonnes removed.
#
#   cover_days       = opening_inventory / consumed_tonnes  -> target recoverable
#                      as opening / cover_days (verified to 1.4e-14)
#   silo_utilisation = closing_inventory / silo_capacity    -> with silo_capacity
#                      also a feature, closing inventory is recoverable
#   deliveries_tonnes-> completes consumed = opening + deliveries - closing, and
#                      is not knowable at forecast time anyway
#
# Validation R2 with them: 0.9958. Without: 0.8765.

feature_cols = [
    "planned_pour_tonnes",
    "opening_inventory_tonnes",
    "rain_mm",
    "avg_temp_c",
    "silo_capacity",
    "site_id",
    "cement_type",
    "region",
    "behavior",
]

X_train = train[feature_cols]
y_train = train[TARGET]

X_val = val[feature_cols]
y_val = val[TARGET]

# X_test / y_test are deliberately not built here - the test split is scored once,
# in 06_Holdout_Validation.ipynb, after model selection is frozen.

print("Features:", len(feature_cols))
print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)

## Data Preprocessing

The dataset contains a mixture of numerical and categorical features. To prepare the data for machine learning, numerical variables are imputed using the median, while categorical variables are imputed using the most frequent category and encoded using one-hot encoding.

These preprocessing steps are combined into a single pipeline to ensure the same transformations are consistently applied during both training and prediction.

In [ ]:
categorical_features = [
    "site_id",
    "cement_type",
    "region",
    "behavior",
]

numerical_features = [
    col for col in feature_cols if col not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        ( "num",SimpleImputer(strategy="median"),
            numerical_features, ),
            
        ( "cat",Pipeline([ ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),]),
            categorical_features, ),])

print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

## Random Forest Model Development

A Random Forest Regressor is trained using the preprocessed dataset. Random Forest is an ensemble learning algorithm that combines multiple decision trees to improve predictive performance and reduce overfitting. The model is trained using the pooled dataset across all sites and will be evaluated on the validation set before comparison with the SARIMAX baseline.

In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model",RandomForestRegressor(
                n_estimators=200,
                max_depth=15,
                min_samples_split=5,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1,),),])

print("Training Random Forest model...")

rf_pipeline.fit(X_train, y_train)

print("Training completed.")

## Model Validation

The trained Random Forest model is evaluated using the validation dataset. Performance is measured using Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and Mean Absolute Percentage Error (MAPE). These metrics provide an objective basis for comparing the machine learning model with the SARIMAX baseline.

In [ ]:
from mig_cement.models.evaluate import evaluate

# Validation predictions
y_pred = rf_pipeline.predict(X_val)

# Evaluate
rf_metrics = evaluate(
    y_true=y_val,
    y_pred=y_pred,
    y_train=y_train,)

print("=" * 60)
print("Random Forest Validation Results")
print("=" * 60)

for metric, value in rf_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

## Test Set Evaluation

After validating the Random Forest model, its performance is assessed on the held-out test dataset. The test set was not used during model training or model selection, providing an unbiased estimate of the model's generalisation performance.

## Hyperparameter Tuning for Random Forest

The baseline Random Forest model is further optimised using hyperparameter tuning. Randomized Search is employed to explore a range of parameter combinations while keeping the computational cost manageable. The objective is to identify the model configuration that produces the best validation performance.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

param_dist = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [10, 15, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"],}

In [ ]:
rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_dist,
    n_iter=15,
    cv=TimeSeriesSplit(n_splits=3),   # review fix: was cv=3 (random KFold)
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
    verbose=2,
)

print("Tuning Random Forest...")

rf_search.fit(X_train, y_train)

print("Done.")

print("Best Parameters")
print(rf_search.best_params_)

print("\nBest CV Score")
print(rf_search.best_score_)

### Validation Performance

In [ ]:
best_rf = rf_search.best_estimator_

y_pred = best_rf.predict(X_val)

best_rf_metrics = evaluate(
    y_true=y_val,
    y_pred=y_pred,
    y_train=y_train,
)

print("=" * 60)
print("Tuned Random Forest Validation Results")
print("=" * 60)

for metric, value in best_rf_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

_Test evaluation moved to `06_Holdout_Validation.ipynb` - the test split is scored once, after model selection is frozen._

## Model Selection

Hyperparameter tuning was performed using RandomizedSearchCV with `TimeSeriesSplit`. The tuned model did not outperform the baseline Random Forest **on validation**, so the baseline Random Forest was retained for comparison against the other models.

Selection is made on validation only; the test split is untouched here.

## LightGBM Model Development

LightGBM is a gradient boosting algorithm designed for high performance on structured datasets. Unlike Random Forest, which builds trees independently, LightGBM constructs trees sequentially, allowing each new tree to correct the errors of the previous ones.

The model is trained using the same preprocessing pipeline and time-based train, validation, and test split as the Random Forest model to ensure a fair comparison.

In [ ]:
from lightgbm import LGBMRegressor

lgbm_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ( "model",LGBMRegressor(
                n_estimators=300,
                learning_rate=0.05,
                num_leaves=31,
                max_depth=10,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                verbosity=-1,),  ), ])

print("Training LightGBM model...")

lgbm_pipeline.fit(X_train, y_train)

print("LightGBM training completed.")

## LightGBM Validation

The trained LightGBM model is evaluated using the validation dataset. The same evaluation metrics used for the Random Forest model are applied to ensure a consistent comparison between the machine learning models.

In [ ]:
y_pred_lgbm = lgbm_pipeline.predict(X_val)

lgbm_val_metrics = evaluate(
    y_true=y_val,
    y_pred=y_pred_lgbm,
    y_train=y_train,)

print("=" * 60)
print("LightGBM Validation Results")
print("=" * 60)

for metric, value in lgbm_val_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

## LightGBM Test Evaluation

The selected LightGBM model is evaluated on the held-out test dataset to assess its ability to generalise to unseen observations. This provides a direct comparison with the Random Forest and SARIMAX models.

## 5.1 Weekly Data Aggregation

Because the forecasting objective is an eight-week horizon, the daily cleaned dataset is aggregated to weekly observations. The aggregation is performed separately for each site and cement type to preserve the individual demand series.

Flow variables such as cement consumption, planned pours, and deliveries are summed over each week, while state and environmental variables are aggregated using appropriate summary statistics.

In [ ]:
# Weekly Data Aggregation
# ============================================

import pandas as pd
import numpy as np

# Make a copy so the original daily dataset remains unchanged
weekly = clean.copy()

# Ensure date is in datetime format
weekly["date"] = pd.to_datetime(weekly["date"])

# Create a week-start column
# Monday is used as the start of the forecasting week
weekly["week"] = weekly["date"].dt.to_period("W-SUN").dt.start_time

# Aggregate daily observations to weekly level
weekly = (weekly.groupby(["week", "site_id", "cement_type", "region", "behavior"],
        as_index=False).agg(
# Target and flow variables → SUM
        consumed_tonnes=("consumed_tonnes", "sum"),
        planned_pour_tonnes=("planned_pour_tonnes", "sum"),
        deliveries_tonnes=("deliveries_tonnes", "sum"),
        rain_mm=("rain_mm", "sum"),

        # State variables → FIRST/LAST
        opening_inventory_tonnes=("opening_inventory_tonnes", "first"),
        silo_capacity=("silo_capacity", "last"),

        # Environmental / utilisation variables → MEAN
        avg_temp_c=("avg_temp_c", "mean"),
        cover_days=("cover_days", "mean"),
        silo_utilisation=("silo_utilisation", "mean"),))

# Sort chronologically
weekly = weekly.sort_values(
    ["site_id", "cement_type", "week"]
).reset_index(drop=True)

print("Weekly dataset shape:", weekly.shape)

print("\nDate range:")
print(weekly["week"].min(), "to", weekly["week"].max())

print("\nNumber of sites:", weekly["site_id"].nunique())
print("Number of cement types:", weekly["cement_type"].nunique())

print("\nWeekly dataset preview:")
display(weekly.head())

## 5.2 Weekly Data Quality Check

Before feature engineering and model training, the weekly dataset is checked for missing values, missing weekly observations, duplicate site-cement-week combinations, and the number of observations available for each demand series.

In [ ]:
# Weekly Data Quality Checks
# ============================================

# 1. Missing values
print("=" * 60)
print("Missing Values")
print("=" * 60)

missing = weekly.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) > 0:
    print(missing)
else:
    print("No missing values found.")


# 2. Check for duplicate site-cement-week combinations
print("\n" + "=" * 60)
print("Duplicate Weekly Observations")
print("=" * 60)

duplicates = weekly.duplicated(
    subset=["week", "site_id", "cement_type"]
).sum()

print("Duplicate rows:", duplicates)


# 3. Number of observations per series
print("\n" + "=" * 60)
print("Observations Per Site-Cement Series")
print("=" * 60)

series_counts = ( weekly
    .groupby(["site_id", "cement_type"])
    .size()
    .describe())

print(series_counts)


# 4. Check missing weeks within each site-cement series
print("\n" + "=" * 60)
print("Missing Weekly Observations")
print("=" * 60)

weekly["week"] = pd.to_datetime(weekly["week"])

missing_week_records = []

for (site, cement), group in weekly.groupby(
    ["site_id", "cement_type"]):
    
    dates = group["week"].sort_values()
    
    expected_weeks = pd.date_range(
        start=dates.min(),
        end=dates.max(),
        freq="7D")
    
    actual_weeks = pd.DatetimeIndex(dates.unique())
    
    missing_weeks = expected_weeks.difference(actual_weeks)
    
    if len(missing_weeks) > 0:
        missing_week_records.append({
            "site_id": site,
            "cement_type": cement,
            "missing_weeks": len(missing_weeks),
            "first_missing_week": missing_weeks.min(),
            "last_missing_week": missing_weeks.max() })


missing_week_df = pd.DataFrame(missing_week_records)

if missing_week_df.empty:
    print("No missing weeks detected within the series.")
else:
    print(
        f"Series with missing weeks: "
        f"{len(missing_week_df)}")
    
    display(
        missing_week_df.head(20) )

## 5.3 Investigating a Missing Week

The identified missing week is checked against the original daily dataset to determine whether the absence represents a genuine zero-activity period or missing data.

In [ ]:
# Check one specific missing week

SITE_CHECK = "SITE_001"
CEMENT_CHECK = "CEM_I"
WEEK_CHECK = pd.Timestamp("2022-01-17")

check = clean[
    (clean["site_id"] == SITE_CHECK) &
    (clean["cement_type"] == CEMENT_CHECK) &
    (clean["date"] >= WEEK_CHECK) &
    (clean["date"] < WEEK_CHECK + pd.Timedelta(days=7))
]

print("Site:", SITE_CHECK)
print("Cement:", CEMENT_CHECK)
print("Week:", WEEK_CHECK.date())
print("\nNumber of daily records found:", len(check))

print("\nDaily records:")
display(check)

## 5.4 Create Complete Weekly Panel

Weeks with no recorded activity are retained in the weekly dataset so that each site-cement demand series has a continuous weekly timeline. This is important for creating lag features and forecasting an eight-week horizon.

In [ ]:
# ============================================
# Create Complete Weekly Panel
# ============================================

# Make sure week is datetime
weekly["week"] = pd.to_datetime(weekly["week"])

# Create complete weekly date range
all_weeks = pd.date_range(
    start=weekly["week"].min(),
    end=weekly["week"].max(),
    freq="7D")

# Get all unique site-cement combinations
series = weekly[
    ["site_id", "cement_type", "region", "behavior"]].drop_duplicates()

# Create every possible week × site × cement combination
complete_index = pd.MultiIndex.from_product(
    [ all_weeks,
        series["site_id"].unique(),
        series["cement_type"].unique()],
    names=["week", "site_id", "cement_type"]).to_frame(index=False)

# Add region and behaviour information
series_info = series.drop_duplicates(
    ["site_id", "cement_type"])

complete_weekly = complete_index.merge(
    series_info,
    on=["site_id", "cement_type"],
    how="left")

# Merge the actual weekly observations
complete_weekly = complete_weekly.merge(
    weekly,
    on=[
        "week",
        "site_id",
        "cement_type",
        "region",
        "behavior"
    ],
    how="left")

# Sort the dataset
complete_weekly = complete_weekly.sort_values(
    ["site_id", "cement_type", "week"]
).reset_index(drop=True)

print("Original weekly rows:", len(weekly))
print("Complete weekly rows:", len(complete_weekly))

print(
    "Expected rows:",
    len(all_weeks) * len(series_info))

display(complete_weekly.head(20))

## 5.5 Handling No-Activity Weeks

Weeks with no recorded activity for a site-cement combination are represented explicitly in the weekly panel. Demand and operational flow variables are set to zero for these periods, while environmental and inventory-related variables are handled separately rather than being replaced indiscriminately with zero.

In [ ]:
# ============================================
# Diagnose Newly Created No-Activity Weeks
# ============================================

# Identify rows where no weekly observation existed
no_activity = complete_weekly["consumed_tonnes"].isna()

print("No-activity weekly rows:", no_activity.sum())

print("\nPercentage of weekly panel:")
print(f"{no_activity.mean() * 100:.2f}%")

print("\nNo-activity rows by site:")
display(
    complete_weekly.loc[no_activity]
    .groupby("site_id")
    .size()
    .sort_values(ascending=False)
    .head(10))

print("\nNo-activity rows by cement type:")
display(
    complete_weekly.loc[no_activity]
    .groupby("cement_type")
    .size()
    .sort_values(ascending=False))

## 5.6 Handling Zero-Activity Weeks

Weekly observations with no recorded activity for a site-cement combination are retained in the forecasting panel. Since no operational record exists for these periods, consumption, planned pour and deliveries are treated as zero. Environmental and inventory variables are not assigned zero values because their absence does not imply zero environmental conditions or zero inventory.

In [ ]:
# ============================================
# Fill Zero-Activity Flow Variables
# ============================================

zero_activity_cols = [
    "consumed_tonnes",
    "planned_pour_tonnes",
    "deliveries_tonnes",]

for col in zero_activity_cols:
    complete_weekly[col] = complete_weekly[col].fillna(0)

print("Remaining missing values:")
print(
    complete_weekly.isna().sum()
    .sort_values(ascending=False))

## 5.7 Reconstruct Weekly Weather Variables

Weekly weather variables are calculated from the original daily cleaned dataset so that weeks with no cement activity still retain the corresponding environmental conditions. Rainfall is aggregated as weekly total precipitation, while temperature is represented by the weekly mean.

In [ ]:
# ============================================
# Reconstruct Weekly Weather
# ============================================

# Make sure date is datetime
clean["date"] = pd.to_datetime(clean["date"])

# Create week-start column
weather = clean.copy()

weather["week"] = (
    weather["date"]
    .dt.to_period("W-SUN")
    .dt.start_time)

# Aggregate weather by week and site
weekly_weather = (
    weather
    .groupby(["week", "site_id"], as_index=False)
    .agg(
        rain_mm=("rain_mm", "sum"),
        avg_temp_c=("avg_temp_c", "mean")))

print("Weekly weather shape:", weekly_weather.shape)

display(weekly_weather.head())

In [ ]:
# Merge weekly weather into the complete panel

complete_weekly = complete_weekly.drop(
    columns=["rain_mm", "avg_temp_c"],
    errors="ignore")

complete_weekly = complete_weekly.merge(
    weekly_weather,
    on=["week", "site_id"],
    how="left")

print("\nMissing values after weather merge:")

print(
    complete_weekly[
        ["rain_mm", "avg_temp_c"]
    ].isna().sum())

## 5.8 Investigating Inventory and Silo Capacity

Inventory and silo-capacity variables are investigated before imputation to determine whether these values are stable at site level and can therefore be recovered for weeks with no activity.

In [ ]:
# ============================================
# Investigate Silo Capacity
# ============================================

print("=" * 60)
print("Silo Capacity by Site")
print("=" * 60)

capacity_check = ( clean
    .groupby(["site_id", "cement_type"])["silo_capacity"]
    .agg(
        count="count",
        unique_values="nunique",
        minimum="min",
        maximum="max")
    .reset_index())

display(capacity_check.head(20))

In [ ]:
# Check whether silo capacity is consistent within each site

site_capacity_check = (
    clean
    .groupby("site_id")["silo_capacity"]
    .nunique())

print("Sites with one unique silo capacity:")
print((site_capacity_check == 1).sum())

print("\nSites with more than one silo capacity:")
print((site_capacity_check > 1).sum())

print("\nMaximum number of unique capacities at a site:")
print(site_capacity_check.max())

## 5.9 Recover Site-Level Silo Capacity

Silo capacity was found to be constant within each site across all cement types and observation periods. Therefore, missing weekly silo-capacity values are recovered using the known capacity for the corresponding site.

In [ ]:
# ============================================
# Recover Site-Level Silo Capacity
# ============================================

# Create a site-level capacity lookup
site_capacity = (
    clean[["site_id", "silo_capacity"]]
    .dropna()
    .drop_duplicates())

# Check that there is only one capacity per site
assert (
    site_capacity.groupby("site_id")["silo_capacity"]
    .nunique()
    .max()
    == 1), "Silo capacity is not constant within at least one site."

# Merge site-level capacity into complete weekly data
complete_weekly = complete_weekly.drop(
    columns=["silo_capacity"],
    errors="ignore")

complete_weekly = complete_weekly.merge(
    site_capacity,
    on="site_id",
    how="left")

print(
    "Missing silo_capacity:",
    complete_weekly["silo_capacity"].isna().sum())

## 5.10 Investigating Opening Inventory

Opening inventory is investigated to determine whether missing values can be reconstructed from the temporal inventory relationship rather than being replaced with arbitrary values.

In [ ]:
# ============================================
# Investigate Opening Inventory
# ============================================

inventory_check = (
    clean
    .groupby(["site_id", "cement_type"])
    .agg(
        observations=("opening_inventory_tonnes", "count"),
        missing=("opening_inventory_tonnes", lambda x: x.isna().sum()),
        minimum=("opening_inventory_tonnes", "min"),
        maximum=("opening_inventory_tonnes", "max")
    )
    .reset_index())

print("=" * 60)
print("Opening Inventory by Site-Cement Series")
print("=" * 60)

display(inventory_check.head(20))

print("\nTotal missing opening inventory:")
print(
    clean["opening_inventory_tonnes"].isna().sum())

In [ ]:
# ============================================
# Check Inventory Balance Equation
# ============================================

inventory_balance = (
    clean["opening_inventory_tonnes"]
    + clean["deliveries_tonnes"]
    - clean["consumed_tonnes"])

difference = (
    clean["closing_inventory_tonnes"]
    - inventory_balance)

print("=" * 60)
print("Inventory Balance Check")
print("=" * 60)

print("Maximum absolute difference:")
print(difference.abs().max())

print("\nMean absolute difference:")
print(difference.abs().mean())

print("\nRows within 0.01 tonnes:")
print(
    (difference.abs() <= 0.01).mean() * 100,
    "%")

## 5.11 Weekly Lag Features

Historical weekly consumption features are created for each site-cement series. These lag features capture recent demand behaviour and provide the machine-learning models with temporal information for weekly demand forecasting.

In [ ]:
# ============================================
# Create Weekly Lag Features
# ============================================

# Ensure correct ordering
complete_weekly = complete_weekly.sort_values(
    ["site_id", "cement_type", "week"]
).reset_index(drop=True)

# Create historical consumption lags
group_cols = ["site_id", "cement_type"]

complete_weekly["consumed_lag_1"] = (
    complete_weekly
    .groupby(group_cols)["consumed_tonnes"]
    .shift(1))

complete_weekly["consumed_lag_2"] = (
    complete_weekly
    .groupby(group_cols)["consumed_tonnes"]
    .shift(2))

complete_weekly["consumed_lag_4"] = (
    complete_weekly
    .groupby(group_cols)["consumed_tonnes"]
    .shift(4))

complete_weekly["consumed_lag_8"] = (
    complete_weekly
    .groupby(group_cols)["consumed_tonnes"]
    .shift(8))

print("Weekly dataset shape:", complete_weekly.shape)

print("\nMissing lag values:")
print(
    complete_weekly[
        [
            "consumed_lag_1",
            "consumed_lag_2",
            "consumed_lag_4",
            "consumed_lag_8"
        ]
    ].isna().sum())

## 5.12 Time-Based Train, Validation and Test Split

A chronological split is used to preserve the temporal structure of the forecasting problem and prevent future observations from influencing model training. The training period covers January 2022 to June 2024, the validation period covers July to September 2024, and the test period covers October to December 2024. The test set is held out until final model selection.

In [ ]:
# ============================================
# Time-Based Train / Validation / Test Split
# ============================================

TRAIN_END = pd.Timestamp("2024-06-30")
VAL_END = pd.Timestamp("2024-09-30")

# Remove rows where lag_8 is unavailable
# These are only the first 8 weeks of each series.
model_data = complete_weekly.dropna(
    subset=["consumed_lag_8"]
).copy()

# Chronological split
train = model_data[
    model_data["week"] <= TRAIN_END].copy()

val = model_data[
    (model_data["week"] > TRAIN_END) &
    (model_data["week"] <= VAL_END)].copy()

test = model_data[
    model_data["week"] > VAL_END].copy()

print("=" * 60)
print("TIME-BASED DATA SPLIT")
print("=" * 60)

for name, part in [
    ("Train", train),
    ("Validation", val),
    ("Test", test)]:
    print(
        f"{name:12s}: {len(part):6,} rows | "
        f"{part['week'].min().date()} -> "
        f"{part['week'].max().date()}")

In [ ]:
# Check that the periods do not overlap

print("\n" + "=" * 60)
print("Date Overlap Check")
print("=" * 60)

print(
    "Train/Validation overlap:",
    len(
        set(train["week"]).intersection(
            val["week"])))

print(
    "Validation/Test overlap:",
    len(set(val["week"]).intersection(
            test["week"])))

print(
    "Train/Test overlap:",
    len(
        set(train["week"]).intersection(
            test["week"]) ))

## 5.13 Weekly Random Forest Model

A global Random Forest regression model is trained on the pooled weekly dataset across all sites and cement types. Site, cement type, region and behaviour are treated as categorical variables, while planned pours, weather conditions and historical consumption lags provide numerical predictors. Inventory-derived variables with unreliable weekly reconstruction are excluded from this first modelling experiment.

In [ ]:
# ============================================
# Weekly Random Forest - Feature Preparation
# ============================================

TARGET = "consumed_tonnes"

feature_cols = [
    "planned_pour_tonnes",
    "rain_mm",
    "avg_temp_c",
    "silo_capacity",
    "consumed_lag_1",
    "consumed_lag_2",
    "consumed_lag_4",
    "consumed_lag_8",
    "site_id",
    "cement_type",
    "region",
    "behavior",
]

X_train = train[feature_cols]
y_train = train[TARGET]

X_val = val[feature_cols]
y_val = val[TARGET]

X_test = test[feature_cols]
y_test = test[TARGET]

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

print("\nTarget shapes:")
print("y_train:", y_train.shape)
print("y_val:  ", y_val.shape)
print("y_test: ", y_test.shape)

In [ ]:
# ============================================
# Check Model Inputs
# ============================================

print("=" * 60)
print("Missing Values in Model Features")
print("=" * 60)

print(X_train.isna().sum())

## 5.14 Weekly Random Forest Training

A global Random Forest regression model is trained using the pooled weekly observations from all site-cement series. Numerical demand, weather and operational features are combined with categorical site, cement type, region and behaviour information. Model performance is first assessed on the validation period, while the test set remains untouched until final model selection.

In [ ]:
# ============================================
# Weekly Random Forest Preprocessor
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Numerical features
numeric_features = ["planned_pour_tonnes","rain_mm","avg_temp_c","silo_capacity",
    "consumed_lag_1", "consumed_lag_2","consumed_lag_4","consumed_lag_8",]

# Categorical features
categorical_features = ["site_id", "cement_type", "region","behavior",]

weekly_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features ),
        ( "cat", OneHotEncoder( handle_unknown="ignore"  ),
            categorical_features),])

In [ ]:
# ============================================
# Train Weekly Random Forest
# ============================================

weekly_rf_pipeline = Pipeline( steps=[("preprocessor", weekly_preprocessor),
 ("model", RandomForestRegressor( n_estimators=300,
                max_depth=15,
                min_samples_split=5,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1, ) ),])

print("Training Weekly Random Forest...")

weekly_rf_pipeline.fit(
    X_train,
    y_train)

print("Weekly Random Forest training completed.")

####Validation

In [ ]:
# ============================================
# Weekly Random Forest Validation
# ============================================

y_val_pred_rf = weekly_rf_pipeline.predict(X_val)

# Prevent negative demand predictions
y_val_pred_rf = y_val_pred_rf.clip(min=0)

weekly_rf_val_metrics = evaluate(
    y_true=y_val,
    y_pred=y_val_pred_rf,
    y_train=y_train,
)

print("=" * 60)
print("Weekly Random Forest - Validation Results")
print("=" * 60)

for metric, value in weekly_rf_val_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

## 5.15 Planned-Pour Benchmark

The planned-pour forecast is used as the primary business benchmark. The benchmark assumes that weekly cement demand is equal to the planned cement pour. The Random Forest model must outperform this benchmark to demonstrate practical forecasting value.

In [ ]:
# ============================================
# Planned-Pour Validation Benchmark
# ============================================

y_val_planned = val["planned_pour_tonnes"].clip(lower=0)

planned_pour_val_metrics = evaluate(
    y_true=y_val,
    y_pred=y_val_planned,
    y_train=y_train,
)

print("=" * 60)
print("Planned-Pour Benchmark - Validation Results")
print("=" * 60)

for metric, value in planned_pour_val_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

In [ ]:
# ============================================
# Compare Random Forest vs Planned Pour
# ============================================

comparison = pd.DataFrame({
    "Random Forest": weekly_rf_val_metrics,
    "Planned Pour": planned_pour_val_metrics,
})

print("=" * 60)
print("Validation Model Comparison")
print("=" * 60)

display(comparison)

## 5.16 Weekly LightGBM Model

A global LightGBM regression model is trained on the same weekly training and validation data used for the Random Forest model. Using identical features and chronological splits allows a direct comparison of the two machine-learning approaches at weekly frequency.

In [ ]:
# ============================================
# Weekly LightGBM Model
# ============================================

from lightgbm import LGBMRegressor

weekly_lgbm_pipeline = Pipeline(
    steps=[("preprocessor", weekly_preprocessor),

        ( "model",LGBMRegressor(
                n_estimators=300,
                learning_rate=0.05,
                num_leaves=31,
                max_depth=10,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                verbosity=-1,) ), ])

print("Training Weekly LightGBM...")

weekly_lgbm_pipeline.fit(
    X_train,
    y_train)

print("Weekly LightGBM training completed.")

In [ ]:
# ============================================
# Weekly LightGBM Validation
# ============================================

y_val_pred_lgbm = weekly_lgbm_pipeline.predict(X_val)

# Prevent negative demand predictions
y_val_pred_lgbm = y_val_pred_lgbm.clip(min=0)

weekly_lgbm_val_metrics = evaluate(
    y_true=y_val,
    y_pred=y_val_pred_lgbm,
    y_train=y_train,)

print("=" * 60)
print("Weekly LightGBM - Validation Results")
print("=" * 60)

for metric, value in weekly_lgbm_val_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

### compare RF AND lightGBM

In [ ]:
# ============================================
# Compare Weekly Models
# ============================================

weekly_model_comparison = pd.DataFrame({
    "Random Forest": weekly_rf_val_metrics,
    "LightGBM": weekly_lgbm_val_metrics,
})

print("=" * 60)
print("Weekly Model Comparison - Validation")
print("=" * 60)

display(weekly_model_comparison)

Weekly data was aggregated into a complete site-cement-week panel. No-activity weeks were identified and handled separately from missing operational values. Weather and site-level silo capacity were reconstructed where appropriate. Inventory-derived variables were excluded from the initial ML experiment because their weekly values could not be reliably reconstructed. Historical consumption lags were created at 1, 2, 4 and 8 weeks. A chronological train/validation/test split was used to evaluate weekly forecasting performance.

## 5.17 Random Forest Hyperparameter Tuning

Random Forest hyperparameters are tuned using time-aware cross-validation on the training data. TimeSeriesSplit is used to preserve chronological ordering and prevent future observations from being used to predict earlier observations. The validation period remains separate and is used only for final model selection.

In [ ]:
# ============================================
# Time-Aware Random Forest Hyperparameter Tuning
# ============================================

from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

# Time-aware cross-validation
tscv = TimeSeriesSplit(n_splits=3)

param_dist = {
    "model__n_estimators": [200, 300, 500, 700],
    "model__max_depth": [10, 15, 20, 25, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", 1.0],
}

rf_search = RandomizedSearchCV(
    estimator=weekly_rf_pipeline,
    param_distributions=param_dist,
    n_iter=15,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
    verbose=2,)

print("Tuning Weekly Random Forest...")

rf_search.fit(
    X_train,
    y_train)

print("Tuning completed.")

print("\nBest Parameters:")
print(rf_search.best_params_)

print("\nBest Time-Series CV RMSE:")
print(-rf_search.best_score_)

In [ ]:
# ============================================
# Tuned Random Forest Validation
# ============================================

tuned_rf = rf_search.best_estimator_

y_val_pred_tuned_rf = tuned_rf.predict(X_val)

y_val_pred_tuned_rf = y_val_pred_tuned_rf.clip(min=0)

tuned_rf_val_metrics = evaluate(
    y_true=y_val,
    y_pred=y_val_pred_tuned_rf,
    y_train=y_train,
)

print("=" * 60)
print("Tuned Weekly Random Forest - Validation")
print("=" * 60)

for metric, value in tuned_rf_val_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

In [ ]:
# ============================================
# Compare Weekly RF Models
# ============================================

rf_tuning_comparison = pd.DataFrame({
    "Original RF": weekly_rf_val_metrics,
    "Tuned RF": tuned_rf_val_metrics,
    "LightGBM": weekly_lgbm_val_metrics,
})

print("=" * 60)
print("Weekly Model Comparison")
print("=" * 60)

display(rf_tuning_comparison)

### Confirm the tuned model and validation data

In [ ]:
# ============================================
# 8-Week Forecast Evaluation
# Step 1: Confirm Model and Validation Data
# ============================================

print("=" * 60)
print("8-WEEK FORECAST EVALUATION SETUP")
print("=" * 60)

print(f"Validation start : {val['week'].min().date()}")
print(f"Validation end   : {val['week'].max().date()}")

print(f"\nValidation rows  : {len(val):,}")
print(f"Validation sites : {val['site_id'].nunique()}")
print(f"Cement types     : {val['cement_type'].nunique()}")

print("\nTuned Random Forest:")
print(rf_search.best_params_)

print("\nValidation feature columns:")
print(X_val.columns.tolist())

## 8-Week Recursive Forecast Evaluation

The tuned Random Forest is evaluated using a recursive multi-step forecasting approach. Forecasts are generated sequentially from one to eight weeks ahead. For each forecast horizon, only information available at the forecasting origin is used.

Historical consumption is used to construct the lag features for the first forecast. For subsequent horizons, previously generated predictions are used when future consumption values are required. This prevents actual future consumption from being used as an input and avoids data leakage.

Performance is evaluated separately for each forecast horizon using WAPE, RMSE, MAE, bias and MASE.

In [ ]:
# ============================================
# Check Validation Weekly Timeline
# ============================================

validation_weeks = (
    val["week"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print("=" * 60)
print("VALIDATION WEEKLY TIMELINE")
print("=" * 60)

print("Number of validation weeks:", len(validation_weeks))
print("\nValidation weeks:")

for i, week in enumerate(validation_weeks, start=1):
    print(f"{i:2d}. {week.date()}")

### Check Required Validation Features

In [ ]:
# ============================================
# 8-Week Forecast Evaluation
# Step 3: Check Required Validation Features
# ============================================

required_features = [
    "planned_pour_tonnes",
    "rain_mm",
    "avg_temp_c",
    "silo_capacity",
    "consumed_lag_1",
    "consumed_lag_2",
    "consumed_lag_4",
    "consumed_lag_8",
    "site_id",
    "cement_type",
    "region",
    "behavior",
]

print("=" * 60)
print("8-WEEK FORECAST FEATURE CHECK")
print("=" * 60)

print("\nMissing values:")
print(val[required_features].isna().sum())

print("\nRequired features present:")
for col in required_features:
    print(f"✓ {col}")

print("\nValidation series:")
print(val.groupby(["site_id", "cement_type"])
       .size()
       .describe()
)

## 8-Week Rolling-Origin Recursive Forecast

The tuned Random Forest is evaluated using rolling-origin recursive forecasting over an eight-week horizon. At each forecasting origin, only historical consumption available up to that point is used to initialise the lag features.

For each subsequent forecast week, the model's previous predictions are added to the historical sequence and used to construct future lag features. This prevents actual future consumption from being used during forecasting.

Future planned-pour and weather variables are taken from the corresponding validation weeks, as these variables are treated as available exogenous information for the forecasting exercise.

Forecast performance is recorded separately for horizons 1 to 8 weeks ahead.

In [ ]:
# ============================================
# 8-WEEK ROLLING-ORIGIN RECURSIVE FORECAST
# ============================================

import pandas as pd
import numpy as np

# --------------------------------------------
# 1. Configuration
# --------------------------------------------

HORIZON = 8

TARGET = "consumed_tonnes"

MODEL_FEATURES = [
    "planned_pour_tonnes",
    "rain_mm",
    "avg_temp_c",
    "silo_capacity",
    "consumed_lag_1",
    "consumed_lag_2",
    "consumed_lag_4",
    "consumed_lag_8",
    "site_id",
    "cement_type",
    "region",
    "behavior",]

GROUP_COLS = ["site_id", "cement_type"]


# --------------------------------------------
# 2. Use the tuned Random Forest
# --------------------------------------------

tuned_rf = rf_search.best_estimator_

print("=" * 60)
print("MODEL USED FOR 8-WEEK FORECAST")
print("=" * 60)

print("Tuned Random Forest")
print(rf_search.best_params_)


# --------------------------------------------
# 3. Prepare historical data
# --------------------------------------------

model_data = complete_weekly.copy()

model_data["week"] = pd.to_datetime(model_data["week"])

model_data = model_data.sort_values(
    GROUP_COLS + ["week"]
).reset_index(drop=True)


# --------------------------------------------
# 4. Forecast origins
#
# We need a complete 8-week horizon.
# The first origin is the last training week.
# --------------------------------------------

train_last_week = train["week"].max()

validation_weeks = (
    val["week"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True))

# Origins must have 8 future validation weeks available

forecast_origins = []

for origin in [train_last_week] + validation_weeks.tolist():

    future_weeks = validation_weeks[  validation_weeks > origin]

    if len(future_weeks) >= HORIZON:forecast_origins.append(origin)


print("\n" + "=" * 60)
print("FORECAST ORIGINS")
print("=" * 60)

for origin in forecast_origins:
    print(origin.date())

print("\nNumber of complete 8-week origins:",
      len(forecast_origins))


# --------------------------------------------
# 5. Storage for forecasts
# --------------------------------------------

forecast_records = []


# --------------------------------------------
# 6. Recursive forecasting
# --------------------------------------------

for origin in forecast_origins:

    print(f"\nForecast origin: {origin.date()}")

    # Eight future validation weeks
    future_weeks = ( validation_weeks[
            validation_weeks > origin]
        .iloc[:HORIZON]
        .tolist())

    # Historical observations available
    # at the forecasting origin
    historical = model_data[
        model_data["week"] <= origin].copy()

    # Loop through each site-cement series
    for (site, cement), hist_group in historical.groupby(
        GROUP_COLS):

        hist_group = hist_group.sort_values("week")

        # Store historical/predicted consumption
        # for this particular series
        consumption_history = list(
            hist_group[TARGET].fillna(0).values)

        # Static information
        region = hist_group["region"].iloc[-1]
        behavior = hist_group["behavior"].iloc[-1]

        # Forecast each future week recursively
        for horizon, forecast_week in enumerate(
            future_weeks,start=1):

            # --------------------------------
            # Future exogenous information
            # --------------------------------

            future_row = val[
                (val["week"] == forecast_week)
                & (val["site_id"] == site)
                & (val["cement_type"] == cement)]

            if future_row.empty:
                continue

            future_row = future_row.iloc[0]

            # --------------------------------
            # Dynamic lag calculation
            # --------------------------------

            def get_lag(lag):
                if len(consumption_history) >= lag:
                    return consumption_history[-lag]
                return np.nan

            lag_1 = get_lag(1)
            lag_2 = get_lag(2)
            lag_4 = get_lag(4)
            lag_8 = get_lag(8)

            # If an 8-week lag is unavailable,
            # this forecast cannot be produced safely.
            if any(
                pd.isna(x)
                for x in [
                    lag_1,
                    lag_2,
                    lag_4,
                    lag_8 ]):
                continue

            # --------------------------------
            # Build model input
            # --------------------------------

            X_future = pd.DataFrame([{ "planned_pour_tonnes":  future_row["planned_pour_tonnes"],
                      "rain_mm": future_row["rain_mm"],
                      "avg_temp_c":future_row["avg_temp_c"],
                      "silo_capacity":future_row["silo_capacity"],
                      "consumed_lag_1":lag_1,
                      "consumed_lag_2":lag_2,
                      "consumed_lag_4":lag_4,
                      "consumed_lag_8":lag_8,
                      "site_id":site,
                      "cement_type":cement,
                      "region":region,
                      "behavior":behavior}])

            # Ensure exact feature order
            X_future = X_future[
                MODEL_FEATURES]

            # --------------------------------
            # Predict
            # --------------------------------

            prediction = tuned_rf.predict(
                X_future
            )[0]

            # Demand cannot be negative
            prediction = max(0, prediction)

            # --------------------------------
            # Actual value
            # --------------------------------

            actual = future_row[TARGET]

            # --------------------------------
            # Save forecast
            # --------------------------------

            forecast_records.append({
                "origin": origin,
                "week": forecast_week,
                "site_id": site,
                "cement_type": cement,
                "horizon": horizon,
                "actual": actual,
                "prediction": prediction})

            # --------------------------------
            # IMPORTANT:
            # Add prediction to history,
            # NOT actual consumption.
            # --------------------------------

            consumption_history.append(prediction)


# --------------------------------------------
# 7. Convert forecasts to DataFrame
# --------------------------------------------

forecast_results = pd.DataFrame(forecast_records)

forecast_results = forecast_results.sort_values(
    ["origin","site_id","cement_type","horizon" ]).reset_index(drop=True)


# --------------------------------------------
# 8. Basic checks
# --------------------------------------------

print("\n" + "=" * 60)
print("8-WEEK FORECAST RESULTS")
print("=" * 60)

print("Forecast rows:",len(forecast_results))

print("Forecast origins:",forecast_results["origin"].nunique())

print( "Series:", forecast_results[["site_id", "cement_type"]
    ].drop_duplicates().shape[0])

print( "Horizons:", sorted( forecast_results["horizon"].unique()))

display(forecast_results.head(20))

In [ ]:
# ============================================
# 8-WEEK FORECAST PERFORMANCE BY HORIZON
# ============================================

print("=" * 60)
print("8-WEEK RECURSIVE FORECAST PERFORMANCE")
print("=" * 60)

horizon_results = []

for h in range(1, 9):

    horizon_data = forecast_results[
        forecast_results["horizon"] == h
    ].copy()

    if horizon_data.empty:
        continue

    metrics = evaluate(
        y_true=horizon_data["actual"],
        y_pred=horizon_data["prediction"],
        y_train=y_train
    )

    results_row = {
        "horizon": f"Week {h}",
        "WAPE": metrics["WAPE"],
        "RMSE": metrics["RMSE"],
        "MAE": metrics["MAE"],
        "bias": metrics["bias"],
        "MAPE_nonzero": metrics["MAPE_nonzero"],
        "MASE": metrics["MASE"],
        "observations": len(horizon_data)
    }

    horizon_results.append(results_row)


horizon_results_df = pd.DataFrame(horizon_results)

print("\n")
display(horizon_results_df)

####  8-WEEK FORECAST VS PLANNED POUR BENCHMARK

In [ ]:
# ============================================
# Add Planned Pour to 8-Week Forecast Results
# ============================================

forecast_results = forecast_results.merge(
    val[
        [ "week", "site_id","cement_type","planned_pour_tonnes" ]],
    on=["week","site_id","cement_type"],how="left")

print("=" * 60)
print("PLANNED POUR ADDED")
print("=" * 60)

print( "Missing planned-pour values:",
    forecast_results["planned_pour_tonnes"].isna().sum())

display(forecast_results.head(10))

### 8-WEEK FORECAST VS PLANNED POUR BENCHMARK

In [ ]:
# ============================================
# 8-WEEK FORECAST VS PLANNED POUR BENCHMARK
# ============================================

benchmark_results = []

for h in range(1, 9):

    horizon_data = forecast_results[
        forecast_results["horizon"] == h
    ].copy()

    # Random Forest metrics
    rf_metrics = evaluate(
        y_true=horizon_data["actual"],
        y_pred=horizon_data["prediction"],
        y_train=y_train)

    # Planned Pour metrics
    planned_metrics = evaluate(
        y_true=horizon_data["actual"],
        y_pred=horizon_data["planned_pour_tonnes"],
        y_train=y_train)

    benchmark_results.append({
        "Horizon": f"Week {h}",

        "RF_WAPE": rf_metrics["WAPE"],
        "Planned_WAPE": planned_metrics["WAPE"],

        "RF_RMSE": rf_metrics["RMSE"],
        "Planned_RMSE": planned_metrics["RMSE"],

        "RF_MAE": rf_metrics["MAE"],
        "Planned_MAE": planned_metrics["MAE"],

        "RF_Bias": rf_metrics["bias"],
        "Planned_Bias": planned_metrics["bias"],

        "RF_MAPE_nonzero": rf_metrics["MAPE_nonzero"],
        "Planned_MAPE_nonzero": planned_metrics["MAPE_nonzero"],

        "RF_MASE": rf_metrics["MASE"],
        "Planned_MASE": planned_metrics["MASE"],

        "Observations": len(horizon_data)
    })


benchmark_results_df = pd.DataFrame(benchmark_results)

print("=" * 80)
print("8-WEEK RANDOM FOREST VS PLANNED POUR")
print("=" * 80)

display(benchmark_results_df)

---

# Random Forest on the SARIMAX Feature Set

Earlier comparisons between SARIMAX and the tree models were not like-for-like: the
SARIMAX runs were weekly per **site**, the ML runs were daily and per
**site x cement type**. Grain alone accounts for most of the apparent gap.

This section removes that confound. Same weekly per-site panel, same 8-week window,
same exogenous features the SARIMAX model was given — only the estimator changes.

Two variants are fitted:

- **exog only** — exactly the SARIMAX regressor set
- **exog + lags** — the fair equivalent, since SARIMAX gets autoregressive terms
  internally and a Random Forest has no such mechanism

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from mig_cement.config import settings

TRAIN_END_W, VAL_END_W = "2024-06-30", "2024-09-30"
HORIZON_WEEKS = 8

fe = pd.read_parquet(settings.processed_dir / "operations_feature_engineered.parquet")
fe["date"] = pd.to_datetime(fe["date"]).dt.to_period("W-SUN").dt.start_time

# same aggregation rules as the SARIMAX weekly experiment
AGG_RF = {
    "y": "sum",
    "planned_pour_tonnes": "sum",
    "planned_pour_next_7": "last",
    "planned_pour_next_14": "last",
    "pour_blocked_rain": "sum",
    "frost": "sum",
    "rain_mm": "mean",
    "avg_temp_c": "mean",
    "opening_inventory_tonnes": "first",
    "inventory_vs_capacity": "first",
    "headroom_tonnes": "first",
    "cover_days_7": "first",
    "days_since_planned_pour": "first",
    "silo_capacity": "first",
    "region": "first",
    "behavior": "first",
}

wk = fe.groupby(["site_id", "date"], as_index=False).agg(AGG_RF)
wk["n_days"] = fe.groupby(["site_id", "date"]).size().values
wk = (wk[wk.n_days == 7].drop(columns="n_days")          # drop partial weeks
        .sort_values(["site_id", "date"]).reset_index(drop=True))

# lags and rolling means - shifted, so no leakage
for lag in (1, 2, 4, 8):
    wk[f"lag_{lag}"] = wk.groupby("site_id").y.shift(lag)
wk["roll_4"] = wk.groupby("site_id").y.transform(lambda s: s.shift(1).rolling(4).mean())
wk["roll_8"] = wk.groupby("site_id").y.transform(lambda s: s.shift(1).rolling(8).mean())

wk = wk.dropna().reset_index(drop=True)
print("weekly per-site panel:", wk.shape)
print("weeks:", wk.date.nunique(), "| sites:", wk.site_id.nunique())

In [ ]:
train_rf = wk[wk.date <= TRAIN_END_W]
val_rf = (wk[(wk.date > TRAIN_END_W) & (wk.date <= VAL_END_W)]
          .groupby("site_id").head(HORIZON_WEEKS))

print(f"train {len(train_rf):5,} rows  {train_rf.date.min().date()} -> {train_rf.date.max().date()}")
print(f"val   {len(val_rf):5,} rows  {val_rf.date.min().date()} -> {val_rf.date.max().date()}"
      f"  ({val_rf.groupby('site_id').size().mean():.0f} weeks per site)")

In [ ]:
# SARIMAX exogenous regressors, verbatim
SARIMAX_EXOG = [
    "planned_pour_tonnes", "planned_pour_next_7", "planned_pour_next_14",
    "pour_blocked_rain", "frost", "rain_mm", "avg_temp_c",
    "opening_inventory_tonnes", "inventory_vs_capacity", "headroom_tonnes",
    "cover_days_7", "days_since_planned_pour",
]
LAGS = ["lag_1", "lag_2", "lag_4", "lag_8", "roll_4", "roll_8"]
CATS = ["site_id", "region", "behavior"]   # SARIMAX gets these free by fitting per site


def fit_rf(features, label, n_estimators=300):
    num = [f for f in features if f not in CATS]
    pre = ColumnTransformer([
        ("num", "passthrough", num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), [f for f in features if f in CATS]),
    ])
    pipe = Pipeline([("preprocessor", pre),
                     ("model", RandomForestRegressor(n_estimators=n_estimators,
                                                     random_state=42, n_jobs=-1))])
    pipe.fit(train_rf[features], train_rf["y"])
    pred = np.clip(pipe.predict(val_rf[features]), 0, None)

    y = val_rf["y"].values
    nz = y != 0
    return {
        "model": label,
        "n_features": len(features),
        "MAPE": np.mean(np.abs((y[nz] - pred[nz]) / y[nz])),
        "RMSE": np.sqrt(np.mean((y - pred) ** 2)),
        "WAPE": np.abs(y - pred).sum() / np.abs(y).sum(),
        "MAE": np.abs(y - pred).mean(),
        "bias": (pred - y).mean(),
    }, pipe

In [ ]:
results_rf = []

m_exog, rf_exog = fit_rf(SARIMAX_EXOG + CATS, "RF (SARIMAX exog only)")
results_rf.append(m_exog)

m_full, rf_full = fit_rf(SARIMAX_EXOG + LAGS + CATS, "RF (SARIMAX exog + lags)")
results_rf.append(m_full)

# SARIMAX figures from the weekly experiments above, same window
results_rf += [
    {"model": "SARIMAX (engineered, weekly)", "n_features": 12,
     "MAPE": 0.0834, "RMSE": 24.6977, "WAPE": 0.0905, "MAE": 15.0323, "bias": 3.5075},
    {"model": "SARIMAX (raw, weekly)", "n_features": 4,
     "MAPE": 0.0974, "RMSE": 27.6384, "WAPE": 0.1059, "MAE": 17.5896, "bias": -0.3019},
]

comparison_rf = pd.DataFrame(results_rf).set_index("model")[
    ["n_features", "MAPE", "RMSE", "WAPE", "MAE", "bias"]]
comparison_rf.round(4)

In [ ]:
imp = pd.Series(
    rf_full.named_steps["model"].feature_importances_[:len(SARIMAX_EXOG + LAGS)],
    index=SARIMAX_EXOG + LAGS,
).sort_values(ascending=False)

print("Random Forest feature importance (SARIMAX exog + lags):")
imp.head(12).to_frame("importance").round(4)

**Like-for-like, the Random Forest beats SARIMAX.** The earlier gap was grain and
features, not the estimator.

| model | MAPE | RMSE | bias |
|---|---|---|---|
| RF (SARIMAX exog only) | **7.1%** | **21.85** | +1.54 |
| RF (SARIMAX exog + lags) | 7.2% | 22.01 | +0.98 |
| SARIMAX (engineered, weekly) | 8.3% | 24.70 | +3.51 |

Three things worth carrying into the report:

- The forward-looking pour columns (`planned_pour_next_7`, `planned_pour_next_14`)
  are what close the gap. The pour schedule is known in advance and is the strongest
  signal in the data; the earlier Random Forest had only backward-looking lags.
- **The lag features add nothing** once the schedule is present — exog-only scores
  marginally better than exog + lags, and `planned_pour_tonnes` alone carries 83% of
  the feature importance. That is consistent with the Step 3 finding of no
  meaningful autocorrelation or seasonality: this is a regression problem with a
  strong exogenous driver, not a classical time-series problem.
- Weather regressors still use their actual validation values, which flatters both
  model families. A schedule-only variant is the number achievable in production.